In [0]:
from pyspark.sql.functions import col, explode
output_path = "/Volumes/workspace/nfl/bronze_landing/"

# Read the JSON file
df_raw = spark.read.option("multiline", "true").json(f"{output_path}*.json")

# get the teams from the json file
df_nfl_teams = (df_raw
    .select(explode("sports").alias("sport")) 
    .select(explode("sport.leagues").alias("league")) # to get to the nfl
    .select(explode("league.teams").alias("team_container")) # to get to the teams  
    .select( # convert JSON to teams table
        col("team_container.team.id").alias("id"),
        col("team_container.team.displayName").alias("displayName"),
        col("team_container.team.abbreviation").alias ("abbreviation"),
        col("team_container.team.color").alias("color")
    ) # create columns 
    .dropDuplicates(["id"])
)

df_nfl_teams.display()

df_nfl_teams.write.mode("overwrite").saveAsTable("workspace.nfl.silver_nfl_teams") # save as table

In [0]:
from pyspark.sql import functions as F

# We only want last week because the last week has the record
df_raw_final = spark.read.option("multiline", "true").json(f"{output_path}*_week18.json")  

# Get the overall record string directly from the nested array
df_season_records = (df_raw_final 
    .select(F.explode("events").alias("games")) 
    # Grab the date right here while "games" is still active!
    .select("games.date", F.explode("games.competitions").alias("matchup")) 
    .select("date", F.explode("matchup.competitors").alias("team_stats")) 
     
    # Now we can cleanly pull the fields out of team stats and use our saved date
    .select( 
        F.year(F.to_timestamp(F.col("date"), "yyyy-MM-dd'T'HH:mm'Z'")).alias("season_year"), 
        F.col("team_stats.id").cast("string").alias("team_id"), 
        F.col("team_stats.records")[0]["summary"].alias("total_record_string") 
    ) 
    .dropDuplicates(["season_year", "team_id"])
)

# Dropping duplicates keeps a single clean snapshot per team per year
df_season_records.display()

In [0]:
# seperate the total record string into wins and losses
df_final_records = df_season_records.withColumn(
    "wins", F.split(F.col("total_record_string"), "-")[0].cast("int")
).withColumn(
    "losses", F.split(F.col("total_record_string"), "-")[1].cast("int")
).withColumn(
    "games_played", (F.col("wins") + F.col("losses")) # check to make sure the games played is correct (17)
).withColumn(
    "win_percentage", F.round((F.col("wins") / F.col("games_played")), 3) # get win percentage
)

df_final_records.display()

df_final_records.write.mode("overwrite").saveAsTable("workspace.nfl.silver_team_season_records") # save as table